# Ramen on EuroSAT: the baseline for the true-label cache experiment

**Question.** How well does Ramen do on EuroSAT, per class and across seeds? This notebook fixes the baseline and
the noise floor that the true-label cache oracle is compared against.

**The experiment this folder builds up to.** Ramen files every test image into the memory queue of its *zero-shot
prediction*, so every zero-shot error lands in the wrong class queue. The oracle `true_label` files each image under
its **true label** instead (it uses ground truth, so it is an upper bound, not a method). Everything else (stored
gradient, retrieval, weighting, step, reset) stays Ramen's. This notebook only covers the baseline.

**Runs in this notebook.**

| Run name | What it is | Uses true labels? |
|---|---|---|
| `NoAdapt` | CLIP zero-shot, no adaptation | no |
| `Ramen` | Ramen (cfg/EuroSAT/Ramen.yaml: capacity 7,500 per class, top-k 50, β 5, signSGD lr 0.01 (cache settings taken from CIFAR-10-C, not tuned on EuroSAT)) | no |

**Data.** EuroSAT RGB (`~/data/EuroSAT/2750`), 27,000 Sentinel-2 satellite images of 64×64 pixels, 10 land-cover
classes (2,000–3,000 images each), **one domain**: the shift is natural photos → satellite images. One shuffled stream
with batch size 100. Model: CLIP ViT-B/16 with the single prompt "a centered satellite photo of {}." (CoOp class names).
With one domain, the per-domain sections are **per true class** here.

**Seeds.** A seed sets the stream order (which images arrive together and in what order). NoAdapt is identical for every seed.
Seed 0 is used for the per-class analysis; seeds 0–4 of Ramen give the noise floor.

**How the runs were produced.** `exp.run(...)` in `new_experiments/ramen_analysis.py` through
`true_label_cache.TrueLabelCacheExperiment` (this folder), with the settings in its `DATASETS` (num_workers changes the stream order, so keep it).
Results are cached in `new_experiments/results/EuroSAT_<model>_mixed_nall_bs100_s<seed>_<run>.pkl`.
A missing run is computed on the GPU. Use `python run_true_label.py EuroSAT <seed>` for a detached run.

**What counts as a result.** A later variant's gain over Ramen counts only if it is above the **noise floor** = 2 × (max − min)
of Ramen's domain-averaged accuracy over seeds 0–4 (computed in section 4).

**How to read this notebook.** 1 Setup · 2 The data · 3 Sanity check of the new method · 4 Ramen results (per class, over the stream, across seeds) ·
5 Interpretation · 6 Glossary. Accuracy is **domain-averaged** (mean of the per-domain accuracies) unless a table says *pooled*.

## 1. Setup

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')  # only used if a run is not cached

import matplotlib.pyplot as plt
import pandas as pd
from true_label_cache import TrueLabelCacheExperiment, seed_table, noise_floor, domain_avg

pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

DATASET, SEED, BY = 'EuroSAT', 0, 'class'
exp = TrueLabelCacheExperiment(DATASET, seed=SEED).run_all(('NoAdapt', 'Ramen'))

## 2. The data the code ran on

Images and classes per domain, and how many the zero-shot model already gets right. These counts are the denominators
for everything below. One domain, so the domain average is the pooled accuracy.

In [ ]:
exp.data_table()

## 3. Sanity check of the new method

`RamenTrueLabel` (src/methods/RamenTrueLabel.py) has a switch `filing`. With `filing: zero_shot` it must be Ramen
exactly, or any later comparison would measure a bug. The smoke test (`python run_true_label.py EuroSAT 0 --smoke <dir>`,
200 images, seed 0:

- `zero_shot_filing` == `Ramen` on 200 of 200 images (100.00%)

(Only two batches. The match of `true_label` with `RamenOracle file_true` was checked on DomainNet, 1,200 images.)

## 4. Ramen results

### 4a. Seed 0, overall

In [ ]:
exp.summary_table(baseline='NoAdapt')

*fixed* = NoAdapt wrong and Ramen right; *broken* = NoAdapt right and Ramen wrong. Net gain = fixed − broken.
One domain: pooled = domain-averaged.

In [ ]:
print(f"pooled accuracy: NoAdapt {exp.results['NoAdapt'].correct.mean():.2%}, Ramen {exp.results['Ramen'].correct.mean():.2%}")

### 4b. Seed 0, per class

In [ ]:
exp.change_table('Ramen', 'NoAdapt', by=BY)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 0.32 * len(exp.change_table('Ramen', by=BY)) + 2.2))
exp.plot_accuracy(['NoAdapt', 'Ramen'], by=BY, ax=axes[0])
axes[0].set_title('Ramen gains on 8 of 10 classes and loses on 2', loc='left', pad=28)
exp.plot_gain('Ramen', 'NoAdapt', by=BY, ax=axes[1])
axes[1].set_title('Two classes carry 60% of the net gain', loc='left')
fig.tight_layout()

### 4b'. Which classes each run predicts

How often each class is predicted, next to how often it is the true class. A class predicted much less often than it
occurs is being absorbed by others.

In [ ]:
exp.prediction_counts(['NoAdapt', 'Ramen'])

### 4c. Seed 0, over the stream

The gain over zero-shot along the stream (rolling window of 1,000 images, pooled over domains).

In [ ]:
ax = exp.plot_stream_gain('Ramen', 'NoAdapt', window=1000)
ax.set_title('The gain shrinks along the stream', loc='left');

### 4d. Seeds 0–4: the noise floor

Ramen's domain-averaged accuracy for each seed (the stream order changes, the data and model do not).

In [ ]:
seeds = seed_table(DATASET, names=('Ramen',))
floor = noise_floor(seeds['Ramen'])
print(f"Ramen over seeds 0-4: {seeds['Ramen'].mean():.2f} ± {seeds['Ramen'].std():.2f}, "
      f"range {seeds['Ramen'].max() - seeds['Ramen'].min():.3f} pts -> noise floor = 2 x range = {floor:.2f} pts")
noadapt = domain_avg(exp.results['NoAdapt'])
seeds.assign(**{'vs NoAdapt (pts)': (seeds['Ramen'] - noadapt).round(2)}).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.2))
ax.plot(seeds['Ramen'], [0] * len(seeds), 'o', ms=9, color='#2a78d6', mec='white', mew=1.5)
for s, v in seeds['Ramen'].items():
    ax.annotate(f'seed {s}', (v, 0), xytext=(0, 9 if s % 2 else -16), textcoords='offset points', ha='center', fontsize=8)
ax.set_yticks([]); ax.spines['left'].set_visible(False)
ax.set_xlabel('Ramen accuracy, domain avg (%)')
ax.set_title(f'Seeds move Ramen by {seeds.Ramen.max() - seeds.Ramen.min():.2f} pts: a gain must clear {floor:.2f}', loc='left');

## 5. Interpretation

- **Ramen beats zero-shot by +14.56 points** (48.07 → 62.63, seed 0), +3,932 of 27,000 images: 5,007 fixed,
  1,075 broken (4.7:1, the best ratio of the three datasets). Ramen changes 33.2% of predictions.
- **Two classes carry most of the gain**: Sea or Lake (+41.3 points, net +1,239, 31.5%) and Forest (+37.3, net +1,120,
  28.5%). Annual Crop Land and Highway or Road add another 31%.
- **Two classes get worse**: Herbaceous Vegetation Land (6.67% → 0.30%, net −191) and Pasture Land (53.6% → 46.5%,
  net −143). Zero-shot predicts Herbaceous Vegetation Land only 531 times for 3,000 true images, so its queue ends with
  531 entries, of which 200 really are Herbaceous (37.7%). After adapting, Ramen predicts it only 21 times. True-label
  filing would give this queue all 3,000 Herbaceous images, so this is where the oracle could change most on this dataset.
- **Seeds**: 61.93–62.68 over seeds 0–4, so the **noise floor is 1.51 points**, the largest of the three. With 10 classes
  and capacity 7,500 per class the memory never evicts (10 × 7,500 > 27,000 images), so it is the stream order
  alone that changes what each image retrieves. A true-label change
  must move EuroSAT by more than 1.51 to count.
- **The gain shrinks along the stream** (pooled gain over zero-shot by thirds of the stream, mean over seeds 0–4):
  +15.2 in the first 9,000 images, +13.7 in the last 9,000, lower at the end on 5 of 5 seeds. The memory never evicts,
  so late images retrieve from a memory full of earlier images filed under their zero-shot prediction. Whether the
  growing queues of absorbing classes (Highway or Road, Annual Crop Land) cause the decline is not tested here.
- No earlier true-label result exists on EuroSAT. In the 200-image smoke test `true_label` was −0.50 (1 image), which says nothing.
- Caveat: per-class numbers are seed 0 only.

## 6. Glossary

| Term | Meaning |
|---|---|
| NoAdapt | CLIP zero-shot classifier, no test-time adaptation |
| Ramen | Test-time adaptation with a per-class memory of (feature, own entropy gradient, entropy). Each image takes one signSGD step on gradients retrieved from the memory, predicts, and the weights are reset |
| queue / filing | Ramen's memory is one fixed-size queue per class; *filing* = choosing which queue a new image is written to (zero-shot prediction in Ramen) |
| true-label filing (`true_label`) | Oracle: file each image under its true class. Uses ground truth, so it is an upper bound, not a method |
| `zero_shot_filing` | `RamenTrueLabel` with filing by zero-shot prediction, which must equal Ramen (sanity check) |
| domain | A part of the stream with one kind of shift: a DomainNet domain, an ImageNet-C corruption; EuroSAT has one |
| domain-averaged accuracy | Mean of the per-domain accuracies (headline convention) |
| pooled accuracy | Correct images / all images |
| fixed / broken | Image wrong under NoAdapt and right under the run / right under NoAdapt and wrong under the run |
| seed | Sets the stream order; NoAdapt does not depend on it |
| noise floor | 2 × (max − min) of the baseline's accuracy over seeds 0–4; a gain must exceed it to count |
| mixed stream | All domains shuffled together into one test stream |